# LightMap: Dataset Analysis and Exploration Notebook

This notebook provides exploratory data analysis, distribution profiling, and structural validation for the low-light restoration benchmark datasets evaluated in the LightMap project:
1. **LOL-v1** (Low-Light Dataset v1)
2. **LOL-v2 Real** (Real captured pairs)
3. **LOL-v2 Synthetic** (Synthetic pairs)
4. **MIT-Adobe FiveK** (Photographic tone adjustment)

> **Note:** Datasets are maintained externally on Google Drive. Run `python scripts/validate_datasets.py` to verify local extraction.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Ensure project root is in sys.path
ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.validate_datasets import get_default_dataset_paths, validate_single_dataset
from scripts.metrics import compute_psnr, compute_ssim, compute_noise_reduction
print("Environment initialized successfully. Root:", ROOT)

## 1. Dataset Availability and Path Scanner

In [ ]:
data_dir = ROOT / "datasets"
dataset_configs = get_default_dataset_paths(data_dir)

print(f"Scanning dataset directory: {data_dir}")
for name, cfg in dataset_configs.items():
    status = validate_single_dataset(name, cfg)
    print(f"{name:18} | Status: {status['status']:14} | Low: {status['low_count']:4} | Normal: {status['normal_count']:4}")

## 2. Low-Light vs. Normal Exposure Histogram Comparison (Synthetic Demo)

In [ ]:
# Generate a synthetic paired example demonstrating severe underexposure
np.random.seed(42)
h, w = 256, 256
normal_img = np.clip(np.random.normal(120, 35, (h, w, 3)), 0, 255).astype(np.uint8)
low_img = np.clip((normal_img.astype(float) / 255.0) ** 2.5 * 255.0 + np.random.normal(0, 4, (h, w, 3)), 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(low_img)
axes[0].set_title(f"Low-Light Exposure (Mean: {np.mean(low_img):.1f})")
axes[0].axis('off')

axes[1].imshow(normal_img)
axes[1].set_title(f"Normal Ground Truth (Mean: {np.mean(normal_img):.1f})")
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 3. Pixel Luminance Distribution

In [ ]:
low_lum = 0.299 * low_img[:, :, 0] + 0.587 * low_img[:, :, 1] + 0.114 * low_img[:, :, 2]
norm_lum = 0.299 * normal_img[:, :, 0] + 0.587 * normal_img[:, :, 1] + 0.114 * normal_img[:, :, 2]

plt.figure(figsize=(8, 4))
plt.hist(low_lum.flatten(), bins=50, alpha=0.6, color='blue', label='Low-Light Input')
plt.hist(norm_lum.flatten(), bins=50, alpha=0.6, color='green', label='Normal Ground Truth')
plt.xlabel('Luminance Value [0-255]')
plt.ylabel('Pixel Frequency')
plt.title('Luminance Distribution Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Benchmark Metric Verification (PSNR / SSIM Baseline)

In [ ]:
psnr_val = compute_psnr(normal_img, low_img)
ssim_val = compute_ssim(normal_img, low_img)
print(f"Unenhanced Low-Light Input Baseline:")
print(f"  - PSNR: {psnr_val:.2f} dB")
print(f"  - SSIM: {ssim_val:.4f}")